# 🎓 AuraTutor - Lớp Học Trực Tuyến 1:1 Thời Gian Thực (Google Colab GPU)

Notebook này cho phép bạn chạy toàn bộ hệ thống gia sư AI tương tác thời gian thực (**Wav2Lip + LiveTalking + WebRTC + EdgeTTS**) trên **GPU Tesla T4 / A100 miễn phí của Google Colab**.

### ⚡ Ưu điểm vượt trội so với chạy trên máy cục bộ:
* **Tốc độ suy luận đạt 60 – 80+ FPS** (gấp 3-4 lần so với card GTX 1050 Ti).
* **16GB VRAM** chuẩn máy chủ, hoàn toàn không nóng máy tính cá nhân, tiết kiệm pin.
* Đường truyền mạng máy chủ Google 1 Gbps giúp gọi WebRTC và Edge-TTS gần như không có độ trễ.
* Có link công khai HTTPS truy cập được từ mọi thiết bị (Laptop, Điện thoại, Máy tính bảng).

---

### Bước 1: Kiểm tra phần cứng GPU (Bắt buộc chọn GPU: Runtime -> Change runtime type -> T4 GPU)

In [ ]:
!nvidia-smi

### Bước 2: Tải mã nguồn AuraTutor và cài đặt thư viện cần thiết

In [ ]:
import os
if not os.path.exists('/content/AuraTutor-LiveTalking'):
    !git clone https://github.com/TontonYuta/AuraTutor-LiveTalking.git /content/AuraTutor-LiveTalking
%cd /content/AuraTutor-LiveTalking
!git pull origin main
!pip install -q -r requirements.txt
!pip install -q edge-tts aiohttp-cors openai python-dotenv


### Bước 3: Tải Model Weights Wav2Lip và Dữ liệu Avatar từ Hugging Face

In [ ]:
# Tải mô hình Wav2Lip (214 MB) và avatar thầy giáo kính trắng (88 MB ZIP)
import os
!mkdir -p models data/avatars

if not os.path.exists('models/wav2lip.pth'):
    print('⏳ Đang tải Wav2Lip checkpoint (214 MB)...')
    !curl -L -o models/wav2lip.pth https://huggingface.co/shibing624/ai-avatar-wav2lip/resolve/main/wav2lip.pth
else:
    print('✅ Model Wav2Lip đã sẵn sàng!')

if not os.path.exists('data/avatars/wav2lip_avatar_glass_man/coords.pkl'):
    print('⏳ Đang tải Avatar thầy giáo kính trắng (88 MB ZIP)...')
    !curl -L -o avatar.zip https://huggingface.co/shibing624/ai-avatar-wav2lip/resolve/main/wav2lip_avatar_glass_man.zip
    !unzip -q -o avatar.zip -d data/avatars/
    !rm -f avatar.zip
else:
    print('✅ Dữ liệu Avatar đã sẵn sàng!')

# Kiểm tra xác thực toàn vẹn 100%
if os.path.exists('data/avatars/wav2lip_avatar_glass_man/coords.pkl') and os.path.exists('models/wav2lip.pth'):
    print('🎉 TẤT CẢ WEIGHTS VÀ DỮ LIỆU ĐÃ SẴN SÀNG 100%!')
else:
    print('❌ CẢNH BÁO: Tải dữ liệu chưa thành công, vui lòng chạy lại ô này!')


### Bước 4: Thiết lập Cloudflare Tunnel (Tạo link truy cập công khai HTTPS)

In [ ]:
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
import os, time, re
from IPython.display import display, HTML

# Khởi động Cloudflare Tunnel ngầm và ghi log an toàn
os.system("pkill cloudflared")
os.system("./cloudflared tunnel --url http://localhost:8010 --logfile /tmp/tunnel.log > /dev/null 2>&1 &")

public_url = None
print("⏳ Đang tạo đường link bảo mật HTTPS công khai...")
for _ in range(25):
    time.sleep(1)
    if os.path.exists("/tmp/tunnel.log"):
        with open("/tmp/tunnel.log") as f:
            content = f.read()
            match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', content)
            if match:
                public_url = match.group(0)
                break

if public_url:
    display(HTML(f'''
    <div style="background:#0f172a; color:#f8fafc; padding:22px; border-radius:12px; border:2px solid #10b981; font-family:sans-serif;">
        <h3 style="color:#10b981; margin-top:0;">🎉 LỚP HỌC AI ĐÃ TẠO XONG VÀ SẴN SÀNG!</h3>
        <p style="font-size:15px;">Nhấp vào đường link dưới đây để mở lớp học trực tuyến 1:1 ngay trên trình duyệt của bạn:</p>
        <p><a href="{public_url}/index.html" target="_blank" style="font-size:18px; font-weight:bold; color:#38bdf8; text-decoration:underline;">👉 {public_url}/index.html</a></p>
        <small style="color:#94a3b8;">⚡ Lưu ý: Hãy giữ cho ô số 5 (bên dưới) tiếp tục chạy để máy chủ duy trì kết nối WebRTC.</small>
    </div>
    '''))
else:
    print("❌ Chưa lấy được link Cloudflare. Vui lòng chạy lại ô này!")

### Bước 5: Khởi động Máy chủ Lớp học AI (Giọng nói Tiếng Việt NamMinhNeural)

In [ ]:
!python app.py \
    --transport webrtc \
    --model wav2lip \
    --avatar_id wav2lip_avatar_glass_man \
    --tts edgetts \
    --REF_FILE vi-VN-NamMinhNeural \
    --stun stun:stun.l.google.com:19302 \
    --listenport 8010